# Burel — Ablazione Test-Time Training (TTT ON vs OFF)

**Cosa fa:** sul tuo `burel_best.pt` (allenato su TinyStories BPE), misura la loss con la memoria che si adatta **ON** vs **OFF**, sulle stesse identiche finestre, su due domini:
- **CONTROL** = TinyStories val (dominio visto)
- **UNSEEN** = codice Python (dominio mai visto) — il terreno dove la tesi "master of expertise on the fly" deve vincere.

**Prima di iniziare:** metti su Google Drive i due file che hai sul Desktop — `burel_ablation.zip` e questo notebook. Poi: Runtime → *Change runtime type* → **GPU (T4)**, e fai partire le celle in ordine.

Il notebook trova **da solo** zip, checkpoint e cache su Drive. Se la ricerca automatica sbaglia, correggi i path nella cella 2.

In [ ]:
# Cella 1 — monta Google Drive
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# Cella 2 — trova zip, checkpoint e cache BPE su Drive (override manuale se serve)
import glob, os

DRIVE = '/content/drive/MyDrive'

# --- override manuali (lasciali None per la ricerca automatica) ---
ZIP_PATH  = None   # es. '/content/drive/MyDrive/burel_ablation.zip'
CKPT_PATH = None   # es. '/content/drive/MyDrive/Burel_checkpoints/burel_best.pt'
DATA_DIR  = None   # cartella con meta.pkl + val.bin + tokenizer.json (la cache TinyStories)

def _first(pattern):
    hits = sorted(glob.glob(pattern, recursive=True))
    return hits

if ZIP_PATH is None:
    hits = _first(f'{DRIVE}/**/burel_ablation.zip')
    print('zip trovati:', hits); ZIP_PATH = hits[0] if hits else None
if CKPT_PATH is None:
    hits = _first(f'{DRIVE}/**/burel_best.pt')
    print('checkpoint trovati:', hits); CKPT_PATH = hits[0] if hits else None
if DATA_DIR is None:
    metas = _first(f'{DRIVE}/**/meta.pkl')
    cand = [os.path.dirname(m) for m in metas
            if os.path.exists(os.path.join(os.path.dirname(m), 'val.bin'))
            and os.path.exists(os.path.join(os.path.dirname(m), 'tokenizer.json'))]
    print('cache BPE candidate (con val.bin+tokenizer.json):', cand)
    DATA_DIR = cand[0] if cand else None

print('\n=> ZIP_PATH =', ZIP_PATH)
print('=> CKPT_PATH =', CKPT_PATH)
print('=> DATA_DIR  =', DATA_DIR)
assert ZIP_PATH and CKPT_PATH and DATA_DIR, 'Manca qualcosa: imposta i path a mano qui sopra.'

In [ ]:
# Cella 3 — scompatta il codice e installa la dipendenza del tokenizer
import zipfile, shutil, os
ROOT = '/content/burel_ablation'
if os.path.exists(ROOT):
    shutil.rmtree(ROOT)
with zipfile.ZipFile(ZIP_PATH) as z:
    z.extractall('/content')
print('scompattato in', ROOT, '->', os.listdir(ROOT))
# torch e' gia' su Colab; serve solo 'tokenizers' per il BPE.
!pip -q install tokenizers

In [ ]:
# Cella 4 — ripristina la cache BPE da Drive in data_cache/ (NIENTE download)
import shutil, os, pickle
CACHE = os.path.join(ROOT, 'data_cache')
os.makedirs(CACHE, exist_ok=True)
for fn in ('meta.pkl', 'val.bin', 'tokenizer.json', 'train.bin'):
    src = os.path.join(DATA_DIR, fn)
    if os.path.exists(src):
        shutil.copy2(src, os.path.join(CACHE, fn))
        print('copiato', fn)
    elif fn != 'train.bin':  # train.bin non serve all'ablazione
        raise FileNotFoundError(f'manca {fn} in {DATA_DIR}')
meta = pickle.load(open(os.path.join(CACHE, 'meta.pkl'), 'rb'))
print('\nmeta:', {k: meta[k] for k in meta if k in ('encoding','vocab_size','tokenizer_file')})
assert meta.get('encoding') == 'bpe', 'Questa NON e\' la cache TinyStories BPE: controlla DATA_DIR.'

In [ ]:
# Cella 5 — costruisci il dominio MAI VISTO = codice Python (il sorgente Burel stesso)
import glob, os
DOMAIN = os.path.join(ROOT, 'domain_code.txt')
py = sorted(glob.glob(os.path.join(ROOT, 'burel', '**', '*.py'), recursive=True)) + \
     sorted(glob.glob(os.path.join(ROOT, 'scripts', '*.py')))
with open(DOMAIN, 'w') as out:
    for p in py:
        out.write(open(p).read() + '\n')
print(f'{len(py)} file -> {DOMAIN} ({os.path.getsize(DOMAIN)} byte di codice)')

In [ ]:
# Cella 6 — LANCIA L'ABLAZIONE (dura un paio di minuti: TTT ON usa il 2° ordine)
import subprocess
cmd = ['python', 'scripts/ablation_ttt.py',
       '--ckpt', CKPT_PATH,
       '--domain_file', DOMAIN,
       '--windows', '128', '--batch', '8']
print('cwd =', ROOT)
print(' '.join(cmd), '\n')
p = subprocess.run(cmd, cwd=ROOT)
print('\nexit code:', p.returncode)

## Come leggere il risultato

Per ogni dominio: tabella `chunk | loss ON | loss OFF | OFF-ON` + tre numeri:
- **benefit = OFF − ON**: se >0, l'adattamento a test-time si paga da solo.
- **extra-adattamento**: di quanto la curva ON scende più ripida della OFF scendendo nella finestra (= si specializza mentre legge).
- **Il confronto che conta:** `benefit(UNSEEN) > benefit(CONTROL)` con `extra-adattamento(UNSEEN) > 0` → la tesi respira.

Lo script stampa già un **VERDETTO**. Incolla tutto l'output nella chat e lo leggiamo insieme.